## Importando Bibliotecas

In [1]:
import pandas as pd
import numpy as np

## Importando Arquivos

In [2]:
df_violencia = pd.read_csv(r"/content/sample_data/violencia_sp.csv", sep=";",low_memory=False)


FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_data/violencia_sp.csv'

## Visualizando os dados

In [ ]:
df_violencia.head(5)

In [ ]:
df_violencia.columns

## Limpeza dos dados

In [ ]:
#Filtrando apenas CS_SEXO = F (Código de sexo da pessoa notificada = Feminino)
df_violencia_tratado = df_violencia[df_violencia["CS_SEXO"] == "F"]

In [ ]:
#Removendo Lixo Estrutural
df_violencia_tratado = df_violencia.drop(columns=["Unnamed: 0"], errors="ignore")

In [ ]:
#Padronizando nome de colunas
df_violencia_tratado.columns = df_violencia_tratado.columns.str.lower().str.strip()

In [ ]:
#Convertendo datas
cols_datas = ["dt_notific", "dt_ocor", "dt_encerra", "dt_obito"]

for col in cols_datas:
    df_violencia_tratado[col] = pd.to_datetime(df_violencia_tratado[col], errors="coerce")

In [ ]:
#Tratamento de valores ausentes na coluna autor_alco (9 = Não informado)
df_violencia_tratado['autor_alco'] = df_violencia_tratado['autor_alco'].replace(9, pd.NA)

In [ ]:
#Ver valores ausentes
df_violencia_tratado.isna().mean().sort_values(ascending=False).head(20)

In [ ]:
#Remover colunas sem dados
df_violencia_tratado = df_violencia_tratado.loc[:, df_violencia_tratado.nunique() >= 1]

## Criando variáveis favoráveis para análise

In [ ]:
#Criando variável de tempo
df_violencia_tratado["ano"] = df_violencia_tratado["dt_ocor"].dt.year
df_violencia_tratado["mes"] = df_violencia_tratado["dt_ocor"].dt.month
df_violencia_tratado["dia_semana"] = df_violencia_tratado["dt_ocor"].dt.dayofweek
df_violencia_tratado = df_violencia_tratado.copy()

In [ ]:
#Criando a variável houve_obito
df_violencia_tratado["houve_obito"] = (
    (df_violencia_tratado["evolucao"] == 2) | #evolução = 2 -> óbito por violência
    (df_violencia_tratado["dt_obito"].notna()) #dados com data de óbito
).astype(int)

#Criando a variável grave baseada em outras variáveis
df_violencia_tratado["grave"] = (
    (df_violencia_tratado["viol_sexu"] == 1) |
    (df_violencia_tratado["viol_tort"] == 1) |
    (df_violencia_tratado["sex_estupr"] == 1) |
    (df_violencia_tratado["houve_obito"] == 1)
).astype(int)

In [ ]:
#Criando variável vitima_em_casa
df_violencia_tratado['vitima_em_casa'] = df_violencia_tratado['local_ocor'].apply(lambda x: 1 if x == 1 else 2)

In [ ]:
# Colunas familiares que existem no DataFrame
familia_cols = ['rel_pai','rel_mae','rel_pad','rel_conj','rel_excon',
                'rel_namo','rel_exnam','rel_filho','rel_irmao']
familia_cols = [col for col in familia_cols if col in df_violencia_tratado.columns]

# Criando função para padronizar: 1 = familiar, 2 = não familiar
def binariza_familia(x):
    try:
        return 1 if int(x) == 1 else 2
    except:
        return 2  # qualquer valor inválido vira 2

# Aplicando função em todas as colunas familiares
for col in familia_cols:
    df_violencia_tratado[col] = df_violencia_tratado[col].apply(binariza_familia)

# Criando coluna agressor_familia: 1 se algum familiar, 2 se nenhum
df_violencia_tratado['agressor_familia'] = df_violencia_tratado[familia_cols].max(axis=1)

# Criando coluna vitima_em_casa (LOCAL_OCOR = 1 significa residência)
df_violencia_tratado['vitima_em_casa'] = df_violencia_tratado['local_ocor'].apply(lambda x: 1 if x == 1 else 2)

In [ ]:
# Criando coluna final_semana
df_violencia_tratado['final_semana'] = df_violencia_tratado['dia_semana'].apply(
    lambda x: 1 if x in [5.0, 6.0] else (2 if pd.notna(x) else pd.NA)
)

In [ ]:
#Criando a variável idade a partir de nu_idade

def converte_idade(valor):
    try:
        valor = int(valor)
        unidade = int(str(valor)[0])
        numero = int(str(valor)[1:])

        if unidade == 1:   # hora
            return numero / (24*365)       # aproximadamente anos
        elif unidade == 2: # dia
            return numero / 365
        elif unidade == 3: # mês
            return numero / 12
        elif unidade == 4: # ano
            return numero
        else:
            return None
    except:
        return None  # valores inválidos ou NaN

# Aplicando no DataFrame
df_violencia_tratado['idade_anos'] = df_violencia_tratado['nu_idade_n'].apply(converte_idade)

# Substituindo NaN por média ou mediana
df_violencia_tratado['idade_anos'] = df_violencia_tratado['idade_anos'].fillna(df_violencia_tratado['idade_anos'].median())

## Selecionando colunas importantes

In [ ]:
#Selecionando colunas
cols = ['grave', 'vitima_em_casa', 'agressor_familia',
               'idade_anos','autor_alco', 'num_envolv','final_semana']
#Criando dataframe apenas com essas colunas
df_modelo = df_violencia_tratado[cols].copy()

In [ ]:
#Verificando tamanho do df
df_modelo.shape

In [ ]:
#Tratando NAs
df_modelo = df_modelo.dropna()

In [ ]:
#Verificando tamanho do df atualizado
df_modelo.shape

In [ ]:
df_modelo.head(5)

## Salvando arquivo para regressão logistica

In [ ]:
df_modelo.to_csv("modelo_limpo.csv", index=False)
#salvando
df_modelo.to_csv(r"/content/sample_data/modelo_limpo.csv", index=False)